In [2]:
import pandas as pd
import numpy as np

# 1. Existing Credit Database
historical_data = {
    'Gender': ['Male', 'Female', np.nan, 'Male', 'Female', 'Male'],
    'Age': [34.5, np.nan, 45.2, 22.0, 51.0, 28.1],
    'Annual_Income': [65000, 82000, np.nan, 38000, 95000, 110000],
    'Cosigner_Collateral': [np.nan, np.nan, 15000, np.nan, np.nan, np.nan] # >50% missing
}
df = pd.DataFrame(historical_data)

print("=== LIVE CREDIT APPLICATION INPUT ===")
print("Leave an answer blank and press Enter to simulate a missing value.\n")

# 2. Collect manual terminal inputs from user
in_gender = input("Enter Applicant Gender (Male/Female): ").strip()
in_age = input("Enter Applicant Age: ").strip()
in_income = input("Enter Annual Income ($): ").strip()
in_collateral = input("Enter Cosigner Collateral ($): ").strip()

# 3. Format inputs (Convert empty strings to true NaNs)
new_applicant = {
    'Gender': in_gender if in_gender != "" else np.nan,
    'Age': float(in_age) if in_age != "" else np.nan,
    'Annual_Income': float(in_income) if in_income != "" else np.nan,
    'Cosigner_Collateral': float(in_collateral) if in_collateral != "" else np.nan
}

# 4. Append new live entry to historical dataframe
df = pd.concat([df, pd.DataFrame([new_applicant])], ignore_index=True)

print("\n=== STEP 1: CALCULATING METRICS ===")
# Generate structural missing profiling table
missing_counts = df.isnull().sum()
missing_pct = df.isnull().mean() * 100

missing_profile = pd.DataFrame({
    'Null Count': missing_counts,
    'Null Percentage (%)': missing_pct
})
print(missing_profile)

print("\n=== STEP 2: APPLYING THRESHOLD FILTER ===")
# Drop columns missing > 50% data across the board
drop_threshold = 50.0
high_null_cols = missing_profile[missing_profile['Null Percentage (%)'] > drop_threshold].index.tolist()
df_filtered = df.drop(columns=high_null_cols)
print(f"Dropped heavy-null columns: {high_null_cols}")

print("\n=== STEP 3: DATA TYPE IMPUTATION ===")
# Dynamically clean the remaining features
for column in df_filtered.columns:
    if df_filtered[column].isnull().sum() > 0:
        if df_filtered[column].dtype == 'object':
            mode_val = df_filtered[column].mode()[0]
            df_filtered[column] = df_filtered[column].fillna(mode_val)
            print(f"-> Filled missing '{column}' values with Mode: {mode_val}")
        else:
            median_val = df_filtered[column].median()
            df_filtered[column] = df_filtered[column].fillna(median_val)
            print(f"-> Filled missing '{column}' values with Median: {median_val}")

print("\n=== STEP 4: VERIFICATION ===")
print("Remaining Nulls:")
print(df_filtered.isnull().sum())

print("\n=== FINAL CLEANED ENTRY ===")
print(df_filtered.tail(1))


=== LIVE CREDIT APPLICATION INPUT ===
Leave an answer blank and press Enter to simulate a missing value.



Enter Applicant Gender (Male/Female):  male
Enter Applicant Age:  29
Enter Annual Income ($):  50000
Enter Cosigner Collateral ($):  50000



=== STEP 1: CALCULATING METRICS ===
                     Null Count  Null Percentage (%)
Gender                        1            14.285714
Age                           1            14.285714
Annual_Income                 1            14.285714
Cosigner_Collateral           5            71.428571

=== STEP 2: APPLYING THRESHOLD FILTER ===
Dropped heavy-null columns: ['Cosigner_Collateral']

=== STEP 3: DATA TYPE IMPUTATION ===
-> Filled missing 'Gender' values with Mode: Male
-> Filled missing 'Age' values with Median: 31.75
-> Filled missing 'Annual_Income' values with Median: 73500.0

=== STEP 4: VERIFICATION ===
Remaining Nulls:
Gender           0
Age              0
Annual_Income    0
dtype: int64

=== FINAL CLEANED ENTRY ===
  Gender   Age  Annual_Income
6   male  29.0        50000.0
